# RU-Guard Phase 5 data-v2 loop: retrain on dataset v2 and export (Colab or Kaggle, free T4)

Same as `notebooks/04_train.ipynb`, but trains `ru-guard-0.5b-v2` and `ru-guard-1.5b-v2` on `data/processed/train_v2.jsonl` (v1.1 + 1,300 ru_pure rows from `training/purify_v2.py`). Val is unchanged. Run all; finished steps are skipped and training resumes after a disconnect.

In [ ]:
# 1. Environment: Colab (Drive) or Kaggle (/kaggle/working). One GPU only.
import os, subprocess
ON_KAGGLE = os.path.exists("/kaggle/working")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
if ON_KAGGLE:
    BASE, REPO = "/kaggle/working", "/kaggle/working/ru-guard"
else:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE, REPO = "/content/drive/MyDrive/ru-guard", "/content/ru-guard"
RUNS, EXPORT, BUILD = f"{BASE}/runs", f"{BASE}/export", "/tmp/gguf_build"
os.makedirs(RUNS, exist_ok=True); os.makedirs(EXPORT, exist_ok=True)
print("kaggle" if ON_KAGGLE else "colab", RUNS, EXPORT)
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True, capture_output=True, text=True).stdout)

In [ ]:
# 2. Helper: stream a shell command and stop on failure.
def run(cmd, cwd=None):
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd or REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"failed with exit code {p.returncode}: {cmd}")

def done(path):
    return os.path.exists(path)

In [ ]:
# 3. Code: clone, or update an existing clone.
if done(REPO):
    run("git pull --ff-only")
else:
    run(f"git clone --depth 1 https://github.com/f243077-cell/Fine-Tuning-Project.git {REPO}", cwd="/")
run("git log --oneline -1")

In [ ]:
# 4. Training stack (versions checked from source, D45). Pip conflict warnings with other preinstalled packages are OK.
run('pip install -q "unsloth==2026.11.4" "unsloth_zoo==2026.11.4" "trl==1.15.0" "transformers==5.17.0" "peft==0.21.3" pyyaml')
run('pip list 2>/dev/null | grep -iE "^(unsloth|unsloth.zoo|trl|transformers|peft|torch|bitsandbytes|datasets|accelerate) "')

In [ ]:
# 5. Dry run on dataset v2. Expect train_rows 5678, train_dropped_long 0.
run("python -m training.train_qlora --config training/configs/qwen2.5-0.5b.yaml --set data.train_path=data/processed/train_v2.jsonl --dry-run")

In [ ]:
# 6. Train and export one model. Skips when the adapter or the export manifest already exists.
def train(config, name, *sets):
    if done(f"{RUNS}/{name}/adapter/adapter_config.json"):
        print(f"skip {name}: adapter exists"); return
    extra = " ".join(f"--set {s}" for s in sets)
    run(f"python -m training.train_qlora --config {config} --out {RUNS} --set run_name={name} {extra}")

def export(name):
    if done(f"{EXPORT}/{name}/export_manifest.json"):
        print(f"skip {name}: exported"); return
    run(f"python -m training.merge_and_export --adapter {RUNS}/{name}/adapter --name {name} --out {EXPORT} --build-dir {BUILD}")
    run(f"ls -lh {EXPORT}/{name}/gguf/")

V2 = "data.train_path=data/processed/train_v2.jsonl"
train("training/configs/qwen2.5-0.5b.yaml", "ru-guard-0.5b-v2", V2)
export("ru-guard-0.5b-v2")

In [ ]:
# 7. ru-guard-1.5b-v2.
train("training/configs/qwen2.5-1.5b.yaml", "ru-guard-1.5b-v2", V2)
export("ru-guard-1.5b-v2")

In [ ]:
# 9. Pack the small logs (v2 runs only).
run("zip -qr train_logs_v2.zip runs/*-v2/train_log.csv runs/*-v2/val_metrics.csv runs/*-v2/run_manifest.json export/*-v2/export_manifest.json && ls -lh train_logs_v2.zip", cwd=BASE)